# Official FD-DINOv2 pooled image evaluation

This notebook computes pooled FD-DINOv2 using Layer 6 AI's official [DGM-Eval](https://github.com/layer6ai-labs/dgm-eval) command-line implementation with DINOv2 ViT-L/14.

All images recursively found below the real and generated roots are staged into flat temporary directories. No class-wise calculation is performed.


## 1. FD-DINOv2-only dependencies

DGM-Eval pins older scientific packages, including NumPy 1.23.3, SciPy 1.9.3, Pillow 9.2.0, and Transformers 4.26.0. Use a dedicated **Python 3.10** kernel for this notebook; its official pins generally fail on Python 3.11+.

Set `INSTALL_DEPS = True` once to clone and install DGM-Eval into this kernel's environment.


In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import tempfile
import time
from datetime import datetime
from pathlib import Path

from IPython.display import display

PYTHON_VERSION_SUPPORTED = (3, 7) <= sys.version_info[:2] <= (3, 10)
print("Python:", sys.version.split()[0])
print("Compatible with official DGM-Eval pins:", PYTHON_VERSION_SUPPORTED)


In [ ]:
INSTALL_DEPS = False
TOOLS_DIR = Path.home() / ".cache" / "duodit-metrics"
DGM_EVAL_DIR = TOOLS_DIR / "dgm-eval"

if INSTALL_DEPS:
    if not PYTHON_VERSION_SUPPORTED:
        raise RuntimeError("Create a Python 3.10 kernel before installing official DGM-Eval dependencies.")
    TOOLS_DIR.mkdir(parents=True, exist_ok=True)
    if not DGM_EVAL_DIR.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/layer6ai-labs/dgm-eval.git",
            str(DGM_EVAL_DIR),
        ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-e", str(DGM_EVAL_DIR)
    ], check=True)
    print("DGM-Eval installed in the current Python 3.10 environment.")
else:
    print("Dependency installation skipped. Set INSTALL_DEPS = True if needed.")


## 2. Evaluation configuration

The official DGM-Eval command downloads DINOv2 ViT-L/14 weights on first use.

DINOv2 tries to use xFormers when it can import it. If xFormers was built for a different PyTorch/CUDA version, DINOv2 can crash with `No operator found for memory_efficient_attention_forward`. Keep `DISABLE_XFORMERS = True` unless you know your xFormers build matches this environment.


In [ ]:
REAL_DIR = Path("/path/to/real")
SAMPLES_DIR = Path("/path/to/samples")

DEVICE = "cuda"
BATCH_SIZE = 32
NUM_WORKERS = 4
DISABLE_XFORMERS = True
MAX_IMAGES = None  # None means all images.

OUTPUT_PATH = Path("official_fd_dinov2.json")
RUN_FD_DINOV2 = False


In [ ]:
IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}


def discover_images(root: Path, maximum: int | None) -> list[Path]:
    if not root.is_dir():
        return []
    paths = sorted(
        path.resolve()
        for path in root.rglob("*")
        if path.is_file() and path.suffix.casefold() in IMAGE_EXTENSIONS
    )
    return paths if maximum is None else paths[:maximum]


real_paths = discover_images(REAL_DIR.expanduser(), MAX_IMAGES)
sample_paths = discover_images(SAMPLES_DIR.expanduser(), MAX_IMAGES)
checks = {
    "python_version_supported": PYTHON_VERSION_SUPPORTED,
    "real_images": len(real_paths),
    "sample_images": len(sample_paths),
    "dgm_eval_checkout": (DGM_EVAL_DIR / "dgm_eval" / "__main__.py").is_file(),
    "xformers_disabled": DISABLE_XFORMERS,
}
display(checks)
ready = (
    PYTHON_VERSION_SUPPORTED
    and len(real_paths) >= 2
    and len(sample_paths) >= 2
    and checks["dgm_eval_checkout"]
)


In [ ]:
fd_dinov2 = None


def stage_images(paths: list[Path], directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    for index, source in enumerate(paths):
        destination = directory / f"{index:08d}{source.suffix.casefold()}"
        try:
            destination.symlink_to(source)
        except OSError:
            os.link(source, destination)


if not RUN_FD_DINOV2:
    print("FD-DINOv2 skipped. Set RUN_FD_DINOV2 = True after preflight passes.")
elif not ready:
    raise ValueError("FD-DINOv2 preflight did not pass.")
else:
    started = time.perf_counter()
    with tempfile.TemporaryDirectory(prefix="official-fd-dinov2-") as temporary_dir:
        temporary_root = Path(temporary_dir)
        staged_real = temporary_root / "real_pooled"
        staged_samples = temporary_root / "samples_pooled"
        stage_images(real_paths, staged_real)
        stage_images(sample_paths, staged_samples)

        command = [
            sys.executable, "-m", "dgm_eval",
            str(staged_real), str(staged_samples),
            "--model", "dinov2",
            "--arch", "vitl14",
            "--metrics", "fd",
            "--device", DEVICE,
            "--batch_size", str(BATCH_SIZE),
            "--num-workers", str(NUM_WORKERS),
            "--nsample", str(max(len(real_paths), len(sample_paths))),
            "--no-load",
        ]
        environment = os.environ.copy()
        if DISABLE_XFORMERS:
            environment["XFORMERS_DISABLED"] = "1"
            environment["XFORMERS_MORE_DETAILS"] = "0"
        completed = subprocess.run(
            command,
            cwd=DGM_EVAL_DIR,
            env=environment,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=False,
        )
        print(completed.stdout)
        if completed.returncode != 0:
            raise RuntimeError(f"Official DGM-Eval failed with exit code {completed.returncode}")
        match = re.search(r"^fd:\s*([-+0-9.eE]+)", completed.stdout, flags=re.MULTILINE)
        if match is None:
            raise ValueError("Could not parse FD-DINOv2 from official output")
        fd_dinov2 = float(match.group(1))

    elapsed = time.perf_counter() - started
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base = OUTPUT_PATH.expanduser().resolve()
    output_path = base.with_name(f"{base.stem}_{timestamp}{base.suffix}")
    payload = {
        "created_at_local": datetime.now().astimezone().isoformat(),
        "metric": "FD-DINOv2",
        "value": fd_dinov2,
        "real_dir": str(REAL_DIR.expanduser().resolve()),
        "samples_dir": str(SAMPLES_DIR.expanduser().resolve()),
        "real_images": len(real_paths),
        "sample_images": len(sample_paths),
        "device": DEVICE,
        "batch_size": BATCH_SIZE,
        "xformers_disabled": DISABLE_XFORMERS,
        "runtime_seconds": elapsed,
        "official_source": "https://github.com/layer6ai-labs/dgm-eval",
        "official_output": completed.stdout,
    }
    output_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    print(f"Official FD-DINOv2: {fd_dinov2:.6f}")
    print(f"Saved: {output_path}")


## Notes

- Keep this metric in its Python 3.10 environment; do not install its old pins into the CMMD or Vendi kernels.
- `DISABLE_XFORMERS = True` is the safest default. It avoids crashes from a stale xFormers wheel while still letting DINOv2 use PyTorch attention.
- DGM-Eval is run directly from its official checkout.
- Compare runs only with the same DINOv2 architecture, official revision, preprocessing, reference set, and image count.
